# S3 Common Operations

In [ ]:
#Upgrade pip to the latest version
!pip3 install --upgrade pip

#Install boto3
!pip3 install boto3

#Import boto3 libraries
import os 
import boto3 
from botocore.client import Config
from boto3 import session

#show Boto3 package version
!pip3 show boto3

In [ ]:
#Create an S3 client

#Define credentials
key_id = os.environ.get('AWS_ACCESS_KEY_ID')
secret_key = os.environ.get('AWS_SECRET_ACCESS_KEY')
endpoint = os.environ.get('AWS_S3_ENDPOINT')
region = os.environ.get('AWS_DEFAULT_REGION')

#Define client session
session = boto3.session.Session(aws_access_key_id=key_id, 
                                aws_secret_access_key=secret_key)

#Define client connection
s3_client = boto3.client('s3', aws_access_key_id=key_id,
                         aws_secret_access_key=secret_key,
                         aws_session_token=None,
                         config=boto3.session.Config(signature_version='s3v4'),
                         endpoint_url=endpoint,
                         region_name=region)

In [ ]:
#List available buckets
s3_client.list_buckets()

In [ ]:
#Print names of available buckets
for bucket in s3_client.list_buckets()['Buckets']:
    print(bucket['Name'])

In [ ]:
#List files in a bucket
bucket_name = 'precip'
s3_client.list_objects_v2(Bucket=bucket_name)

In [ ]:
#Print only names of files
bucket_name = 'precip'
response = s3_client.list_objects_v2(Bucket=bucket_name)

# Using .get() provides an empty list if 'Contents' doesn't exist
for obj in response.get('Contents', []):
    print(obj['Key'])

In [ ]:
bucket_name = 'precip'
for key in s3_client.list_objects_v2(Bucket=bucket_name)['Contents']:
    print(key['Key'])

In [ ]:
#Upload file to bucket
s3_client.upload_file('test.csv', 'precip', 'test.csv')

In [ ]:
#List files to verify file has been uploaded
for key in s3_client.list_objects_v2(Bucket='precip')['Contents']: 
    print(key['Key'])

In [ ]:
#Upload file to bucket
s3_client.upload_file('test.csv', 'labels', 'test.csv')

In [ ]:
#Download file from bucket
# s3_client.download_file('labels', 'labels/AO0045466_6105_2021-10.tif', 'AO0045466_6105_2021-10.tif')

bucket_name = 'labels' # 'precip' # or 'labels' depending on your bucket setup
local_folder = './labels'
s3_key = 'labels/AO0045466_6105_2021-10.tif' # Added quotes here

os.makedirs(local_folder, exist_ok=True)
local_file = os.path.join(local_folder, 'AO0045466_6105_2021-10.tif')

# Pass s3_key as a string
s3_client.download_file(bucket_name, s3_key, local_file)

In [ ]:
#Copy files between buckets
copy_source = {
    'Bucket': 'labels',
    'Key': 'labels/AO0045466_6105_2021-10.tif'
}
s3_client.copy(copy_source, 'precip', 'labels/AO0045466_6105_2021-10.tif')

In [ ]:
#Copy files between buckets
copy_source = {
    'Bucket': 'precip',
    'Key': 'test.csv'
}
s3_client.copy(copy_source, 'labels', 'test.csv')

In [ ]:
#Delete file from bucket
s3_client.delete_object(Bucket='labels', Key='labels/AO0045466_6105_2021-10.tif')

In [ ]:
#Delete file from bucket
s3_client.delete_object(Bucket='precip', Key='test.csv')

In [ ]:
#List files to verify that the file has been deleted
bucket_name = 'precip'
for key in s3_client.list_objects_v2(Bucket=bucket_name)['Contents']:
    print(key['Key'])

In [ ]:
#Delete bucket
s3_client.delete_bucket(Bucket='labels')

In [ ]:
#List buckets to verify that the bucket has been deleted
for bucket in s3_client.list_buckets()['Buckets']:
    print(bucket['Name'])

In [ ]:
## Installing AWS S3 CLI

## Installing AWS S3 CLI

In [ ]:
# From the Workbench Terminal, run (NOT HERE!!!)
curl "https://awscli.amazonaws.com/awscli-exe-linux-x86_64.zip" -o "awscliv2.zip"
unzip awscliv2.zip
./aws/install -i ~/.local/aws-cli -b ~/.local/bin

In [ ]:
!aws s3 ls --endpoint-url=https://minio-s3-adleo-b29e08.apps.shift.nerc.mghpcc.org

## Installing Rclone CLI

In [ ]:
# From the Workbench Terminal, run (NOT HERE!!!)
curl -O https://downloads.rclone.org/rclone-current-linux-amd64.zip
unzip rclone-current-linux-amd64.zip
cd rclone-*-linux-amd64
mkdir -p ~/.local/bin
cp rclone ~/.local/bin/
chmod +x ~/.local/bin/rclone

In [ ]:
# !rclone ls remote:path
!rclone lsd "nerc:"

In [ ]:
!rclone ls "nerc:precip"

In [ ]:
## From Remote (S3) to Local

In [ ]:
#!rclone sync nerc:<public-bucket-name> <local-folder> --progress --dry-run
#!rclone sync nerc:<public-bucket-name> <local-folder> --progress
!rclone sync nerc:precip myfolder --progress --dry-run

In [ ]:
## From Local to Remote (S3)

In [ ]:
#!rclone sync <local-folder> nerc:<your-bucket-name> --progress --dry-run
#!rclone sync <local-folder> nerc:<your-bucket-name> --progress
!rclone sync myfolder nerc:precip --progress --dry-run

In [ ]:
## From Remote (S3) to Local

In [ ]:
#!rclone copy nerc:<public-bucket-name> <local-folder> --progress --dry-run
#!rclone copy nerc:<public-bucket-name> <local-folder> --progress
!rclone copy nerc:labels/images/AO0045466_2021-10.tif myfolder --progress --dry-run

In [ ]:
## From Local to Remote (S3)

In [ ]:
#!rclone copy <local-folder> nerc:<your-bucket-name> --progress --dry-run
#!rclone copy <local-folder> nerc:<your-bucket-name> --progress
!rclone copy myfolder nerc:precip --progress --dry-run